# Extension — half-life of the actual Uniswap L2−L1 basis

The H1/H2 half-life (≈ 10 s) is estimated on a Binance proxy. `dex_midprices.parquet` contains both pools' prices, so the cross-chain basis can be built directly: $P^{L1}_t = 10^{12}/m^{eth}_t$, $P^{L2}_t = 10^{12}\thinspace  m^{arb}_t$, $b^{U}_t = P^{L2}_t - P^{L1}_t$. The half-life is $\tau_{1/2} = \Delta t\thinspace \ln 2 / (-\ln\varphi)$ from an AR(1) fit on a grid of step $\Delta t$.

*Exploratory: no costs, no depth, quote times are irregular (median gap ≈ 2.6 s), and forward-filling stale quotes biases AR(1) persistence upward. The estimate is therefore checked on several grids.*

In [1]:
import pandas as pd
from IPython.display import Image, display

pd.options.display.float_format = "{:,.4f}".format


def show(path):
    display(Image(filename=str(path)))

In [2]:
from crosschain_arb import checks
from crosschain_arb.halflife import ar1_half_life
from crosschain_arb.io import data_path, load_latency
from crosschain_arb.prices import resample_basis, to_usd

usd = to_usd(pd.read_parquet(data_path("dex_midprices.parquet")))
print(usd[["price_l1", "price_l2", "basis"]].describe(percentiles=[0.01, 0.5, 0.99]).T)

                  count       mean      std        min         1%        50%  \
price_l1 2,749,292.0000 1,785.4421 131.7553 1,375.8697 1,538.1501 1,811.7120   
price_l2 2,749,292.0000 1,785.3052 131.7972 1,374.0823 1,537.5350 1,811.5309   
basis    2,749,292.0000    -0.1369   1.9250   -72.1052    -4.9089    -0.1633   

                99%        max  
price_l1 2,104.4724 2,138.6269  
price_l2 2,104.4724 2,137.1304  
basis        4.6147    53.0502  


In [3]:
rows = []
for freq, dt in [("1s", 1.0), ("5s", 5.0), ("30s", 30.0)]:
    hl = ar1_half_life(resample_basis(usd, freq), dt_seconds=dt, resample=False)
    rows.append({"grid": freq, "phi": hl.phi, "half_life_s": hl.half_life_s})
grid = pd.DataFrame(rows)
grid

,grid,phi,half_life_s
0,1s,0.9959,169.4321
1,5s,0.9796,168.0459
2,30s,0.8776,159.2579


Across grids the half-life of the Uniswap L2−L1 basis is about **160–170 s**, far above the ≈ 10 s of the proxy.

## Compared with observed bridge latency

In [4]:
half_life_1s = grid.loc[grid.grid == "1s", "half_life_s"].iloc[0]
shares = checks.latency_fit_checks(
    load_latency(), {"Uniswap basis half-life": half_life_1s, "proxy half-life (9.97 s)": 9.97}
)
{k: f"{100 * v:.2f}%" for k, v in shares["share_faster_than_half_life"].items()}

{'Uniswap basis half-life': '71.96%', 'proxy half-life (9.97 s)': '0.01%'}

72% of the observed transfers settle faster than the Uniswap basis's half-life, against 0.01% for the proxy's. The comparison uses the full export of transfers and ignores costs, depth and quote staleness, so it is indicative only. Next step: repeat H1/H2 on this series.